# Corrected modeling on the combined data

This notebook starts from the already-created combined input/output table. It does not repeat file copying, trajectory feature extraction, or table merging.

Main corrections compared with the earlier modeling cells:

- Use the combined data file: `chr3/engineered_feature_table.csv`.
- Use explicit input and target column lists instead of column positions.
- Remove input columns that were normalized by `area_um2`, because that leaks the answer when predicting nucleus area.
- Exclude metadata/coordinate targets like `nucleus_idx`, `frame`, `x_nm`, and `y_nm`.
- Split and cross-validate by `nucleus_id`, so loci from the same nucleus do not appear in both train and test.
- The combined CSV does not contain an `experiment` or `batch` column, so `nucleus_id` is the strongest available grouping variable here.
- Drop missing values per target instead of requiring every target to be present for every row.
- Include a mean-prediction baseline, so the ML models have to beat a simple reference.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [10]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_CV_SPLITS = 5
RF_N_ITER = 8

# These candidates make the notebook work whether it is launched from
# traditional_ml/, trajectory_to_nuclear_features/, or the repo root.
DATA_CANDIDATES = [
    Path("../data/chr3/engineered_feature_table.csv"),
    Path("data/chr3/engineered_feature_table.csv"),
    Path("trajectory_to_nuclear_features/data/chr3/engineered_feature_table.csv"),
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the combined modeling table. Tried: "
        + ", ".join(str(path) for path in DATA_CANDIDATES)
    )

OUTPUT_DIR = DATA_PATH.parent / "traditional_ml_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using data: {DATA_PATH}")
print(f"Writing outputs to: {OUTPUT_DIR}")

Using data: chr3/engineered_feature_table.csv
Writing outputs to: chr3/traditional_ml_results


In [11]:
df = pd.read_csv(DATA_PATH)

print("Rows, columns:", df.shape)
print("Unique nuclei:", df["nucleus_id"].nunique())
print("Rows per nucleus:")
print(df.groupby("nucleus_id").size().value_counts().sort_index())

display(df.head())

Rows, columns: (757, 47)
Unique nuclei: 358
Rows per nucleus:
1    132
2    101
3     82
4     38
5      5
Name: count, dtype: int64


,nucleus_id,locus_id,x_variance_nm2,y_variance_nm2,mean_step_size_nm,max_step_size_nm,displacement_variance_nm2,total_path_length_nm,net_displacement_nm,straightness_index,...,frame,x_nm,y_nm,dist_to_membrane_nm,dist_to_centroid_nm,norm_radial_pos,local_intensity_mean,local_intensity_std,nuc_mean_intensity,local_to_nuc_ratio
0,U2OS_chr3_195M-488+195.7M-565+198M-647_RNP1_H3...,G_loci2,43283.790196,1.936018e+06,438.506718,575.367906,6563.359420,4385.067182,4170.542233,0.951078,...,6.0,10799.369091,7971.480909,2126.501818,6613.834545,0.243382,104.377273,1.931818,104.499091,0.998836
1,U2OS_chr3_195M-488+195.7M-565+198M-647_RNP1_H3...,G_loci2,100248.910080,5.094996e+05,495.591985,757.875928,47726.556780,2477.959925,2113.995971,0.853120,...,6.0,15647.301818,10645.581818,1295.079091,5329.550909,0.187900,104.129091,2.060000,103.540909,1.005691
2,U2OS_chr3_195M-488+195.7M-565+198M-647_RNP1_H3...,G_loci1,62587.416507,1.887476e+05,230.390090,574.430490,35788.925055,1612.730633,1505.046290,0.933229,...,6.0,24392.963636,34497.354545,-6917.577273,18128.543636,0.000000,NaN,NaN,NaN,NaN
3,U2OS_chr3_195M-488+195.7M-565+198M-647_RNP1_H3...,G_loci1,91664.518885,2.423805e+06,502.243881,744.320718,20007.273806,5022.438805,4672.700717,0.930365,...,6.0,19834.908182,10892.479091,3801.290909,7359.780000,0.340055,105.588182,2.133636,104.866364,1.006909
4,U2OS_chr3_195M-488+195.7M-565+198M-647_RNP1_H3...,G_loci2,55543.787184,9.560874e+05,518.240346,732.283552,26659.138381,3627.682421,2673.911628,0.737085,...,6.0,24762.022727,27075.881818,1678.517273,12116.139091,0.121427,107.802727,2.440909,104.866364,1.027964


## Column choices

The input columns below are trajectory-derived movement summaries. The three normalized columns from the older notebook are deliberately not used because they include `area_um2` in their calculation.

The targets are split into nucleus-level measurements and locus/local-environment measurements. Metadata and raw coordinate targets are listed separately and excluded from modeling.

In [12]:
GROUP_COL = "nucleus_id"

INPUT_COLS = [
    "x_variance_nm2",
    "y_variance_nm2",
    "mean_step_size_nm",
    "max_step_size_nm",
    "displacement_variance_nm2",
    "total_path_length_nm",
    "net_displacement_nm",
    "straightness_index",
    "turning_angle_mean",
    "turning_angle_var",
    "turning_angle_median",
    "turning_angle_acf_lag1",
    "turning_angle_acf_lag2",
    "speed_autocorr_lag1",
    "vector_autocorr_lag1",
    "step_size_psd_total_power",
    "step_size_psd_peak_frequency",
    "step_size_psd_spectral_centroid",
]

LEAKY_INPUTS_NOT_USED = [
    "convex_hull_area_normalized_nuc_area_10_neg_6",
    "convex_hull_area_normalized_nuc_area_frames_10_neg_6",
    "radius_of_gyration_normalized_area_10_neg_3",
]

NUCLEUS_TARGETS = [
    "area_um2",
    "perimeter_px",
    "circularity",
    "bbox_h",
    "bbox_w",
    "eccentricity",
    "major_axis_px",
    "minor_axis_px",
    "orientation_deg",
    "solidity",
    "nuc_intensity_mean",
    "nuc_intensity_std",
    "nuc_intensity_med",
]

LOCUS_LOCAL_TARGETS = [
    "dist_to_membrane_nm",
    "dist_to_centroid_nm",
    "norm_radial_pos",
    "local_intensity_mean",
    "local_intensity_std",
    "nuc_mean_intensity",
    "local_to_nuc_ratio",
]

TARGETS_NOT_USED = [
    "nucleus_idx",
    "frame",
    "x_nm",
    "y_nm",
]

TARGET_COLS = NUCLEUS_TARGETS + LOCUS_LOCAL_TARGETS

required_cols = [GROUP_COL] + INPUT_COLS + TARGET_COLS
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}")

present_leaky_cols = [col for col in LEAKY_INPUTS_NOT_USED if col in df.columns]
present_excluded_targets = [col for col in TARGETS_NOT_USED if col in df.columns]

print(f"Input features used: {len(INPUT_COLS)}")
print(INPUT_COLS)
print(f"Targets used: {len(TARGET_COLS)}")
print(TARGET_COLS)
print("Leaky input columns present but not used:", present_leaky_cols)
print("Metadata/coordinate targets present but not used:", present_excluded_targets)

Input features used: 18
['x_variance_nm2', 'y_variance_nm2', 'mean_step_size_nm', 'max_step_size_nm', 'displacement_variance_nm2', 'total_path_length_nm', 'net_displacement_nm', 'straightness_index', 'turning_angle_mean', 'turning_angle_var', 'turning_angle_median', 'turning_angle_acf_lag1', 'turning_angle_acf_lag2', 'speed_autocorr_lag1', 'vector_autocorr_lag1', 'step_size_psd_total_power', 'step_size_psd_peak_frequency', 'step_size_psd_spectral_centroid']
Targets used: 20
['area_um2', 'perimeter_px', 'circularity', 'bbox_h', 'bbox_w', 'eccentricity', 'major_axis_px', 'minor_axis_px', 'orientation_deg', 'solidity', 'nuc_intensity_mean', 'nuc_intensity_std', 'nuc_intensity_med', 'dist_to_membrane_nm', 'dist_to_centroid_nm', 'norm_radial_pos', 'local_intensity_mean', 'local_intensity_std', 'nuc_mean_intensity', 'local_to_nuc_ratio']
Leaky input columns present but not used: ['convex_hull_area_normalized_nuc_area_10_neg_6', 'convex_hull_area_normalized_nuc_area_frames_10_neg_6', 'radius_

In [5]:
model_df = df.replace([np.inf, -np.inf], np.nan).copy()
model_df = model_df.loc[model_df[GROUP_COL].notna()].reset_index(drop=True)

all_groups = model_df[GROUP_COL].astype(str).to_numpy()
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(model_df, groups=all_groups))

train_groups = set(all_groups[train_idx])
test_groups = set(all_groups[test_idx])
assert train_groups.isdisjoint(test_groups)

pd.DataFrame({GROUP_COL: sorted(test_groups)}).to_csv(
    OUTPUT_DIR / "heldout_test_nuclei.csv",
    index=False,
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train nuclei:", len(train_groups))
print("Test nuclei:", len(test_groups))
print(f"Saved held-out nucleus IDs to {OUTPUT_DIR / 'heldout_test_nuclei.csv'}")

Train rows: 810
Test rows: 220
Train nuclei: 182
Test nuclei: 46
Saved held-out nucleus IDs to chr3/traditional_ml_results/heldout_test_nuclei.csv


In [13]:
def make_group_cv(groups, max_splits=MAX_CV_SPLITS):
    n_groups = pd.Series(groups).nunique()
    n_splits = min(max_splits, n_groups)
    if n_splits < 2:
        raise ValueError("Need at least two groups for grouped cross-validation.")
    return GroupKFold(n_splits=n_splits)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    std = np.std(y_true, ddof=0)
    mean_abs = np.mean(np.abs(y_true))
    pred_std = np.std(y_pred, ddof=0)
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": rmse,
        "nrmse": rmse / std if std > 0 else np.nan,
        "mae": mae,
        "nmae": mae / mean_abs if mean_abs > 0 else np.nan,
        "corr": np.corrcoef(y_true, y_pred)[0, 1] if pred_std > 0 and std > 0 else np.nan,
    }


def prepare_target_data(target):
    target_df = model_df[[GROUP_COL, target] + INPUT_COLS].copy()
    target_df = target_df.loc[target_df[target].notna()].reset_index(drop=True)

    is_test = target_df[GROUP_COL].astype(str).isin(test_groups)
    train_df = target_df.loc[~is_test].reset_index(drop=True)
    test_df = target_df.loc[is_test].reset_index(drop=True)

    X_train = train_df[INPUT_COLS]
    y_train = train_df[target].to_numpy(float)
    groups_train = train_df[GROUP_COL].astype(str).to_numpy()

    X_test = test_df[INPUT_COLS]
    y_test = test_df[target].to_numpy(float)
    groups_test = test_df[GROUP_COL].astype(str).to_numpy()

    return X_train, y_train, groups_train, X_test, y_test, groups_test


def top_features_from_search(search, n=5):
    model = search.best_estimator_.named_steps["model"]
    if hasattr(model, "coef_"):
        scores = np.abs(model.coef_)
    elif hasattr(model, "feature_importances_"):
        scores = model.feature_importances_
    else:
        return ""

    return ", ".join(
        pd.Series(scores, index=INPUT_COLS)
        .sort_values(ascending=False)
        .head(n)
        .index
    )

In [14]:
def make_elastic_net_search(groups):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(max_iter=10000, random_state=RANDOM_STATE)),
    ])

    param_grid = {
        "model__alpha": np.logspace(-4, 1, 20),
        "model__l1_ratio": [0.1, 0.5, 0.9],
    }

    return GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=make_group_cv(groups),
        scoring="r2",
        n_jobs=-1,
        refit=True,
    )


def make_random_forest_search(groups):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
    ])

    param_distributions = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 3, 5],
        "model__max_features": ["sqrt", "log2", 0.5, 0.75],
    }

    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=RF_N_ITER,
        cv=make_group_cv(groups),
        scoring="r2",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
    )

## Train and evaluate

For each target, the notebook:

1. Keeps rows where that target is available.
2. Uses the same held-out nucleus groups for test evaluation.
3. Tunes Elastic Net and Random Forest with grouped cross-validation on the training nuclei only.
4. Evaluates the final refit model on the held-out nuclei.
5. Compares against a simple training-mean baseline.

In [15]:
results = []

for target in TARGET_COLS:
    X_train, y_train, groups_train, X_test, y_test, groups_test = prepare_target_data(target)

    n_train_groups = pd.Series(groups_train).nunique()
    n_test_groups = pd.Series(groups_test).nunique()

    if len(y_train) == 0 or len(y_test) == 0:
        print(f"Skipping {target}: no train or test rows after filtering.")
        continue

    if n_train_groups < 2:
        print(f"Skipping {target}: fewer than two training nuclei.")
        continue

    if np.unique(y_train).size <= 1:
        print(f"Skipping {target}: constant training target.")
        continue

    print(f"\nTarget: {target}")
    print(f"Train rows/groups: {len(y_train)}/{n_train_groups}; test rows/groups: {len(y_test)}/{n_test_groups}")

    # Baseline: predict the training mean. This gives a sanity-check reference.
    dummy_cv = make_group_cv(groups_train)
    dummy_X_train = np.zeros((len(y_train), 1))
    dummy_X_test = np.zeros((len(y_test), 1))
    dummy_cv_pred = cross_val_predict(
        DummyRegressor(strategy="mean"),
        dummy_X_train,
        y_train,
        groups=groups_train,
        cv=dummy_cv,
    )
    dummy = DummyRegressor(strategy="mean").fit(dummy_X_train, y_train)
    dummy_test_pred = dummy.predict(dummy_X_test)
    dummy_cv_metrics = regression_metrics(y_train, dummy_cv_pred)
    dummy_test_metrics = regression_metrics(y_test, dummy_test_pred)

    results.append({
        "target": target,
        "model": "mean_baseline",
        "n_train_rows": len(y_train),
        "n_test_rows": len(y_test),
        "n_train_groups": n_train_groups,
        "n_test_groups": n_test_groups,
        "cv_r2": dummy_cv_metrics["r2"],
        "test_r2": dummy_test_metrics["r2"],
        "test_rmse": dummy_test_metrics["rmse"],
        "test_nrmse": dummy_test_metrics["nrmse"],
        "test_mae": dummy_test_metrics["mae"],
        "test_nmae": dummy_test_metrics["nmae"],
        "test_corr": dummy_test_metrics["corr"],
        "best_params": "{}",
        "top_features": "",
    })

    for model_name, make_search in [
        ("elastic_net", make_elastic_net_search),
        ("random_forest", make_random_forest_search),
    ]:
        search = make_search(groups_train)
        search.fit(X_train, y_train, groups=groups_train)
        y_test_pred = search.predict(X_test)
        test_metrics = regression_metrics(y_test, y_test_pred)

        results.append({
            "target": target,
            "model": model_name,
            "n_train_rows": len(y_train),
            "n_test_rows": len(y_test),
            "n_train_groups": n_train_groups,
            "n_test_groups": n_test_groups,
            "cv_r2": search.best_score_,
            "test_r2": test_metrics["r2"],
            "test_rmse": test_metrics["rmse"],
            "test_nrmse": test_metrics["nrmse"],
            "test_mae": test_metrics["mae"],
            "test_nmae": test_metrics["nmae"],
            "test_corr": test_metrics["corr"],
            "best_params": json.dumps(search.best_params_, sort_keys=True),
            "top_features": top_features_from_search(search),
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["target", "model"]).reset_index(drop=True)

results_path = OUTPUT_DIR / "grouped_model_comparison.csv"
results_df.to_csv(results_path, index=False)

print(f"\nSaved results to {results_path}")
display(results_df)


Target: area_um2
Train rows/groups: 810/182; test rows/groups: 220/46



Target: perimeter_px
Train rows/groups: 810/182; test rows/groups: 220/46

Target: circularity
Train rows/groups: 810/182; test rows/groups: 220/46

Target: bbox_h
Train rows/groups: 810/182; test rows/groups: 220/46

Target: bbox_w
Train rows/groups: 810/182; test rows/groups: 220/46

Target: eccentricity
Train rows/groups: 810/182; test rows/groups: 220/46

Target: major_axis_px
Train rows/groups: 810/182; test rows/groups: 220/46

Target: minor_axis_px
Train rows/groups: 810/182; test rows/groups: 220/46

Target: orientation_deg
Train rows/groups: 810/182; test rows/groups: 220/46

Target: solidity
Train rows/groups: 810/182; test rows/groups: 220/46

Target: nuc_intensity_mean
Train rows/groups: 810/182; test rows/groups: 220/46

Target: nuc_intensity_std
Train rows/groups: 810/182; test rows/groups: 220/46

Target: nuc_intensity_med
Train rows/groups: 810/182; test rows/groups: 220/46

Target: dist_to_membrane_nm
Train rows/groups: 810/182; test rows/groups: 220/46

Target: dist_

,target,model,n_train_rows,n_test_rows,n_train_groups,n_test_groups,cv_r2,test_r2,test_rmse,test_nrmse,test_mae,test_nmae,test_corr,best_params,top_features
0,area_um2,elastic_net,810,220,182,46,-0.011223,1.318193e-02,133.846993,0.993387,92.291081,0.278210,1.301070e-01,"{""model__alpha"": 5.455594781168514, ""model__l1...","max_step_size_nm, straightness_index, turning_..."
1,area_um2,mean_baseline,810,220,182,46,-0.009035,-9.194743e-05,134.744187,1.000046,94.260166,0.284146,-3.835283e-17,{},
2,area_um2,random_forest,810,220,182,46,-0.022401,-2.126801e-02,136.163259,1.010578,94.148122,0.283808,4.316472e-02,"{""model__max_depth"": 5, ""model__max_features"":...","displacement_variance_nm2, mean_step_size_nm, ..."
3,bbox_h,elastic_net,810,220,182,46,-0.017561,-3.483438e-03,30.825174,1.001740,26.195556,0.222236,6.717336e-17,"{""model__alpha"": 5.455594781168514, ""model__l1...","x_variance_nm2, y_variance_nm2, mean_step_size..."
4,bbox_h,mean_baseline,810,220,182,46,-0.006593,-3.483438e-03,30.825174,1.001740,26.195556,0.222236,6.717336e-17,{},
5,bbox_h,random_forest,810,220,182,46,-0.090308,-1.426272e-02,30.990291,1.007106,26.253800,0.222730,5.319506e-02,"{""model__max_depth"": 5, ""model__max_features"":...","straightness_index, max_step_size_nm, mean_ste..."
6,bbox_w,elastic_net,810,220,182,46,-0.024449,-2.171366e-09,31.311162,1.000000,20.274725,0.176232,-3.445203e-16,"{""model__alpha"": 5.455594781168514, ""model__l1...","x_variance_nm2, y_variance_nm2, mean_step_size..."
7,bbox_w,mean_baseline,810,220,182,46,-0.010492,-2.171366e-09,31.311162,1.000000,20.274725,0.176232,-3.445203e-16,{},
8,bbox_w,random_forest,810,220,182,46,-0.059844,-3.311738e-02,31.825411,1.016424,21.285860,0.185021,-1.113243e-02,"{""model__max_depth"": 5, ""model__max_features"":...","x_variance_nm2, mean_step_size_nm, step_size_p..."
9,circularity,elastic_net,810,220,182,46,-0.032974,-2.659259e-05,0.079099,1.000013,0.051712,0.096287,5.104049e-16,"{""model__alpha"": 0.012742749857031334, ""model_...","x_variance_nm2, y_variance_nm2, mean_step_size..."


In [16]:
best_by_target = (
    results_df
    .loc[results_df["model"] != "mean_baseline"]
    .sort_values(["target", "test_r2"], ascending=[True, False])
    .groupby("target", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_path = OUTPUT_DIR / "best_model_by_target.csv"
best_by_target.to_csv(best_path, index=False)

print(f"Saved best-model summary to {best_path}")
display(best_by_target[["target", "model", "cv_r2", "test_r2", "test_nrmse", "test_corr", "top_features"]])

Saved best-model summary to chr3/traditional_ml_results/best_model_by_target.csv


,target,model,cv_r2,test_r2,test_nrmse,test_corr,top_features
0,area_um2,elastic_net,-0.011223,1.318193e-02,0.993387,1.301070e-01,"max_step_size_nm, straightness_index, turning_..."
1,bbox_h,elastic_net,-0.017561,-3.483438e-03,1.001740,6.717336e-17,"x_variance_nm2, y_variance_nm2, mean_step_size..."
2,bbox_w,elastic_net,-0.024449,-2.171366e-09,1.000000,-3.445203e-16,"x_variance_nm2, y_variance_nm2, mean_step_size..."
3,circularity,elastic_net,-0.032974,-2.659259e-05,1.000013,5.104049e-16,"x_variance_nm2, y_variance_nm2, mean_step_size..."
4,dist_to_centroid_nm,random_forest,-0.024559,3.340776e-04,0.999833,1.126658e-01,"step_size_psd_spectral_centroid, speed_autocor..."
5,dist_to_membrane_nm,random_forest,0.027492,1.689527e-01,0.911618,4.432912e-01,"displacement_variance_nm2, step_size_psd_spect..."
6,eccentricity,elastic_net,-0.041243,-4.422378e-03,1.002209,9.151725e-16,"x_variance_nm2, y_variance_nm2, mean_step_size..."
7,local_intensity_mean,random_forest,0.079319,2.680898e-02,0.986504,4.144463e-01,"displacement_variance_nm2, max_step_size_nm, s..."
8,local_intensity_std,random_forest,0.104682,2.052710e-02,0.989683,3.456856e-01,"displacement_variance_nm2, max_step_size_nm, s..."
9,local_to_nuc_ratio,random_forest,0.025844,-3.147825e-02,1.015617,-2.749038e-02,"displacement_variance_nm2, step_size_psd_total..."


## How to read the output

`grouped_model_comparison.csv` has one row per target and model. The most important columns are:

- `cv_r2`: grouped cross-validation R2 on the training nuclei. For Elastic Net and Random Forest, this is the best grouped-CV score from hyperparameter tuning.
- `test_r2`: R2 on nuclei that were never used for training or tuning.
- `test_nrmse`: test RMSE divided by the standard deviation of the held-out target. Smaller is better.
- `test_corr`: correlation between true and predicted target values on the held-out nuclei.
- `mean_baseline`: a reference model that predicts the training-set average for that target.

A useful model should usually beat `mean_baseline` on the held-out test nuclei. If `test_r2` is near or below zero, the model is not doing better than predicting a simple average for that target.